# Week 3-4: S3 Data Engineering Pipeline

This notebook reproduces the Titanic ETL pipeline from the lecture:

1. Download the raw Titanic dataset.
2. Upload the raw CSV to AWS S3.
3. Download the raw CSV back from S3 for processing.
4. Clean and engineer the data with pandas.
5. Upload the cleaned CSV back to S3.
6. Verify that the cleaned file can be queried from S3.

Security note: do not commit real AWS credentials to GitHub. Fill the credential values only inside your private Colab runtime.

In [ ]:
# Install dependencies in Google Colab if needed.
!pip install boto3 pandas seaborn matplotlib -q

In [ ]:
import boto3
import pandas as pd
import urllib.request
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

## 1. AWS Configuration

Fill in your AWS credentials and S3 bucket name before running the notebook in Colab.

Do not push a notebook containing real keys to GitHub.

In [ ]:
AWS_ACCESS_KEY_ID = ""
AWS_SECRET_ACCESS_KEY = ""
REGION_NAME = "eu-north-1"
BUCKET_NAME = ""

s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION_NAME,
)

## 2. Extract: Download Raw Titanic Data

The raw dataset is downloaded from the public GitHub source used in the lecture.

In [ ]:
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
raw_filename = "titanic_raw.csv"

urllib.request.urlretrieve(url, raw_filename)
print(f"Downloaded raw dataset: {raw_filename}")

## 3. Load Raw Data To S3

This step uploads the raw CSV to your S3 bucket. This preserves a raw copy before transformation.

In [ ]:
print(f"Uploading '{raw_filename}' to S3 bucket '{BUCKET_NAME}'...")
s3_client.upload_file(raw_filename, BUCKET_NAME, raw_filename)
print("Raw upload complete.")

## 4. Download Raw Data From S3 For Processing

The lecture pipeline reads the raw data back from S3 before cleaning. This proves that S3 is part of the data pipeline.

In [ ]:
downloaded_filename = "downloaded_titanic.csv"

print("Downloading raw data from S3 for processing...")
s3_client.download_file(BUCKET_NAME, raw_filename, downloaded_filename)

df = pd.read_csv(downloaded_filename)
print(f"Original Data Shape: {df.shape}")
display(df.head())

## 5. Inspect Missing Values

Before cleaning, inspect which columns contain missing values.

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

## 6. Transform: Clean And Engineer Features

Cleaning decisions:

- Fill missing `Age` values with the median age.
- Fill missing `Embarked` values with the most common port.
- Drop `Cabin` because it has many missing values.
- Convert `Sex` and `Embarked` into dummy variables.

In [ ]:
df_clean = df.copy()

df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())
df_clean["Embarked"] = df_clean["Embarked"].fillna(df_clean["Embarked"].mode()[0])
df_clean.drop(columns=["Cabin"], inplace=True)

df_clean = pd.get_dummies(df_clean, columns=["Sex", "Embarked"], drop_first=True)

for column in ["Sex_male", "Embarked_Q", "Embarked_S"]:
    if column in df_clean.columns:
        df_clean[column] = df_clean[column].astype(int)

print(f"Clean Data Shape: {df_clean.shape}")
display(df_clean.head())

## 7. Quick Visual Check

This plot gives a simple view of the survival distribution after cleaning.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_clean, x="Survived", color="steelblue")
plt.title("Titanic Survival Count")
plt.xlabel("Survived")
plt.ylabel("Passenger Count")
plt.show()

## 8. Load Clean Data Back To S3

Save the engineered dataset locally and upload it to S3.

In [ ]:
clean_filename = "titanic_clean.csv"
df_clean.to_csv(clean_filename, index=False)

print(f"Uploading engineered data ('{clean_filename}') back to S3...")
s3_client.upload_file(clean_filename, BUCKET_NAME, clean_filename)
print("Engineered upload complete.")

## 9. Verify Clean Data From S3

Download the cleaned dataset from S3 again to verify that it was uploaded successfully.

In [ ]:
verification_filename = "titanic_verification.csv"
s3_client.download_file(BUCKET_NAME, clean_filename, verification_filename)

df_verify = pd.read_csv(verification_filename)
print("Successfully queried edited data from S3.")
display(df_verify.head())